In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pydub import AudioSegment
import wave
import math
import soundfile as sf
import librosa
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift, Shift
from pathlib import Path
from tqdm import tqdm
from IPython.display import Audio, display
import shutil
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from collections import defaultdict
from PIL import Image
import gc
import psutil

c:\Users\shang\Programs\anaconda3\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Defining functions

In [ ]:
# a function to print the properties of an audio file
def f_get_properties(filename): 
    with wave.open(filename, 'rb') as wav_file:
        num_channels = wav_file.getnchannels()
        sample_rate = wav_file.getframerate()
        sample_width = wav_file.getsampwidth()
        num_frames = wav_file.getnframes()
        duration = num_frames / float(sample_rate)
        bit_depth = sample_width * 8  # Sample width is in bytes
    return num_channels, sample_rate, bit_depth, duration

# Preprocessing data for the analysis

Converting audio files to format to .wav  
And resampling all WAVs to:
- Mono
- 44.1 kHz
- 16-bit PCM

## Defining which audio recordings could be used for "other" class

In [ ]:
df = pd.read_csv('data/data_20250330.csv') # Update the file path as needed !!!

# Create a new column to get unique identifiers for each row
df['Unique'] = df['ID'].astype(str) + '_' + df['Month'].astype(str) + '_' + df['Day'].astype(str) + '_' + df['Time'].astype(str)

df = df[df['Habitant'] == 1].reset_index(drop=True)  # Keep only rows where Habitant is 1

df = df[['Common name', 'Confidence', 'Unique', 'ID']]

df.info()

In [ ]:
# adding a check column to mark rows with "hume's warbler"
df['Check'] = (df['Common name'] == "hume's warbler").astype(int)

marked_uniques = df.loc[df['Check'] == 1, 'Unique'].unique()

# Fill out the 'Check' column based on marked uniques
df.loc[df['Unique'].isin(marked_uniques), 'Check'] = 1

# choosing the location where no "hume's warbler" was found
other = df[df['Check'] == 0]



In [ ]:
other = other[other['Confidence'] > 0.5] # adjust level as needed
other.info()

In [ ]:
(other['Common name'].value_counts(normalize=True)*100).round(3)

Define which unique audio recordings can be used

In [ ]:
samples = other.drop_duplicates(subset='Unique').reset_index(drop=True)  # Remove duplicates based on 'Unique' column

In [ ]:
samples['Common name'].nunique() 

In [ ]:
samples.to_csv('data/other.csv', index=False)  # Save the filtered samples to a new CSV file

Working with splitted audio set that is used for "other" classification only

Exploring random buzz and call sounds

In [ ]:
data_paths = {
    'buzz': 'audio_data/buzz/1.WAV',
    'song': 'audio_data/call/0.WAV'
} 


In [ ]:
# Checking the properties of the specified audio files in data_path
for condition, filepath in data_paths.items():
    num_channels, sample_rate, bit_depth, duration = f_get_properties(filepath)
    print(f"Condition: {condition}")

    filename = os.path.basename(filepath)

    print(f"File: {filename}")
    print(f"Number of Channels: {num_channels}")
    print(f"Sample Rate: {sample_rate} Hz")
    print(f"Bit Depth: {bit_depth}-bit")
    print(f"Duration: {duration:.2f} seconds")
    print("-------------------------------")


Let's play each audio sample

In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=44100)
    print(f"Name of the sample: {key}")
    display(Audio(x, rate=sr))

Visual inspection of waveform of each sample

In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=44100)
    
    # Generate time values for the x-axis
    time = librosa.times_like(x, sr=sr)
    
    plt.figure(figsize=(12, 4))
    plt.plot(time, x, label='Waveform', linewidth=2)
    plt.title(f'Waveplot of {key}')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.show()

Zooming in on the data

In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=44100)
    
    # Zoom in on a specific range
    n0 = 400
    n1 = 800
    plt.figure(figsize=(12, 4))
    plt.plot(x[n0:n1])
    plt.title(f'Zoomed-in Waveform of {key}')
    plt.xlabel('Sample Index')
    plt.ylabel('Amplitude')
    plt.grid()
    plt.show()

Creating a spectrogram to visualize the frequency content of the signal over time  
The color intensity in the spectrogram represents the amplitude of different frequencies at different time points

In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=44100)
    
    # Compute the Short-Time Fourier Transform (STFT)
    X = librosa.stft(x)
    
    # Convert magnitude spectrogram to decibels
    Xdb = librosa.amplitude_to_db(abs(X))
    
    # Plot the spectrogram
    plt.figure(figsize=(12, 4))
    librosa.display.specshow(Xdb, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar()
    plt.title(f'Spectrogram of {key}')
    plt.show()


Displaying a spectrogram with a logarithmic frequency scale

In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=44100)
    
    # Compute the Short-Time Fourier Transform (STFT)
    X = librosa.stft(x)
    
    # Convert magnitude spectrogram to decibels
    Xdb = librosa.amplitude_to_db(abs(X))
    
    # Plot the spectrogram
    plt.figure(figsize=(12, 4))
    librosa.display.specshow(Xdb, sr=sr, x_axis='time', y_axis='log')
    plt.colorbar()
    plt.title(f'Spectrogram of {key}')
    plt.show()

Visualization of Mel-frequency cepstral coefficients (MFCCs)  

MFCCs are coefficients representing the short-term power spectrum of a sound signal.  

The MFCCs capture the spectral characteristics of the audio signal and are particularly useful for capturing features related to human perception of sound.

In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=44100)
    
    # Compute the mel spectrogram
    S = librosa.feature.melspectrogram(y=x, sr=sr, n_mels=128, fmax=8192)

    # Convert the mel spectrogram to MFCCs
    mfccs = librosa.feature.mfcc(S=librosa.power_to_db(S), n_mfcc=13)

    plt.figure(figsize=(12, 4))
    plt.title(f'MFCCs of {key}')
    librosa.display.specshow(mfccs, sr=sr, x_axis='time')
    plt.colorbar(format='%+2.0f dB')
    plt.show()

Generating a chromagram.  
Visual representation of the distribution of pitch content over time in the audio signal.  
The resulting plot is a useful tool for analyzing the harmonic content and tonal characteristics of the audio.

In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=None)
# Set the hop length
    hop_length = 12

# Compute the chromagram
    chromagram = librosa.feature.chroma_stft(y=x, sr=sr, hop_length=hop_length)

# Plot the chromagram
    plt.figure(figsize=(12, 4))
    librosa.display.specshow(chromagram, x_axis='time', y_axis='chroma', hop_length=hop_length, cmap='coolwarm')
    plt.title(f"Chromagram of {key}")
    plt.colorbar()
    plt.show()


In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=None)

# Compute the chromagram
    chromagram = librosa.feature.chroma_cens(y=x)

# Plot the chromagram
    plt.figure(figsize=(12, 4))
    librosa.display.specshow(chromagram, x_axis='time', y_axis='chroma', cmap='coolwarm')
    plt.colorbar()
    plt.title('Chroma CENS')
    plt.show()